In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")

In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm, trange
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN, THINK_START_TOKEN



os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [5]:
# task_name = "plan_generation_po"
# domain_name = f"blocksworld_mystery"
# eval_results = load_dataset_from_file(domain_name, task_name)["instances"]
# eval_results = {x["dataset_idx"]: x for x in eval_results}

In [6]:
tokenizer = initialize_tokenizer(model_id)

In [73]:
dataset = load_dataset(f"dmitriihook/qwq-32b-planning-6-blocks")["train"]
dataset_high = load_dataset("dmitriihook/blocksworld-6-qwq-reasoning-parts-high")["train"]
dataset_low = load_dataset("dmitriihook/blocksworld-6-blocks-qwq-reasoning-parts-low-v4")["train"]

(…)6-blocks-qwq-reasoning-parts-low-v4.json:   0%|          | 0.00/9.83M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [8]:
model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=compute_dtype, attn_implementation="sdpa", 
                                                device_map="auto")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
import ray
ray.init(address="auto", namespace="blocksworld")

2025-03-26 23:31:45,572	INFO worker.py:1654 -- Connecting to existing Ray cluster at address: 10.61.4.10:6379...
2025-03-26 23:31:45,583	INFO worker.py:1832 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


Python version:,3.11.11
Ray version:,2.42.1
Dashboard:,http://127.0.0.1:8265


(raylet) The autoscaler failed with the following error:
Terminated with signal 15
  File "/home/nebius/openr1/lib/python3.11/site-packages/ray/autoscaler/_private/monitor.py", line 719, in <module>
    monitor.run()
  File "/home/nebius/openr1/lib/python3.11/site-packages/ray/autoscaler/_private/monitor.py", line 604, in run
    self._run()
  File "/home/nebius/openr1/lib/python3.11/site-packages/ray/autoscaler/_private/monitor.py", line 458, in _run
    time.sleep(AUTOSCALER_UPDATE_INTERVAL_S)



In [10]:
actor_handle = ray.get_actor("dataset_actor")
hidden_states = ray.get(actor_handle.get_hidden_states.remote())

In [11]:
n_rows = 300
layer = 39

In [74]:
import re

labels_low = [
    "initial-state-understanding",
    "goal-state-understanding",
    "state-tracking",
    "action-exploration",
    "state-tracking"
]

def parse_sections(text:str, labels: list[str]) -> list[tuple[str, str]]:
    pattern = r'\["([^"]+)"\](.*?)\["end-section"\]'
    matches = re.findall(pattern, text, re.DOTALL)
    
    sections = []
    for label, content in matches:
        if label in labels:
            sections.append((label, content.strip()))
    
    return sections

In [75]:
def map_sections_into_tokens(sections: list[tuple[str, str]], row: dict) -> list[dict]:
    # tokens = tokenize_blocksworld_generation(tokenizer, row)
    generation = row["generation"]
    section_tokens = []
    for label, content in sections[1:]:
        text_pos = generation.find(content[:300])
        if text_pos == -1:
            continue
        text_before = generation[:text_pos]
        tokens_before = tokenize_blocksworld_generation(tokenizer, row, text_before)[0, :-2]
        content_tokens = tokenizer.encode(" " + content)[:-5]
        section_tokens.append({
            "label": label,
            "pos_before": len(tokens_before),
            "pos_after": len(tokens_before) + len(content_tokens),
            "text_pos": text_pos,
            "content": content
        })

    return section_tokens
def test_sections():
    idx = 10
    generation = dataset[idx]["generation"]
    sections = parse_sections(dataset_low[idx]["label"], labels_low)
    section_tokens = map_sections_into_tokens(sections, dataset[idx])
    tokens = tokenize_blocksworld_generation(tokenizer, dataset[idx])[0]

    for st in section_tokens:
        s = generation[st["text_pos"]:st["text_pos"]+300]
        dt = tokenizer.decode(tokens[st["pos_before"]:st["pos_before"]+300])
        inter_len = min(len(s[:200]), len(dt))
        if s[:inter_len] != dt[:inter_len]:
            print("****"*10)
            # print()
            print(tokenizer.tokenize(s[:inter_len]))
            print()
            print("----"*10)
            print(tokenizer.tokenize(dt[:inter_len]))
            print()


test_sections()

****************************************
['So', 'Ġthe', 'Ġfinal', 'Ġstack', 'Ġwould', 'Ġbe', 'ĠC', 'Ġ->', 'ĠB', 'Ġ->', 'ĠF', 'Ġ->', 'ĠD', 'Ġ->', 'ĠA', 'Ġ->', 'ĠE', '.', 'ĠSo', 'ĠE', 'Ġis', 'Ġnow', 'Ġon', 'Ġtop', ',', 'Ġbut', 'Ġoriginally', 'Ġit', 'Ġwas', 'Ġthe', 'Ġbase', '.', 'ĠSo', 'Ġwe', 'Ġneed', 'Ġto', 'Ġmove', 'ĠE', 'Ġfrom', 'Ġthe', 'Ġbottom', 'Ġto', 'Ġthe', 'Ġtop', '.', 'ĠTo', 'Ġdo', 'Ġthat', ',', 'Ġwe', 'Ġhave', 'Ġto', 'Ġdismantle', 'Ġthe', 'Ġexist', 'i']

----------------------------------------
['Ġthe', 'Ġfinal', 'Ġstack', 'Ġwould', 'Ġbe', 'ĠC', 'Ġ->', 'ĠB', 'Ġ->', 'ĠF', 'Ġ->', 'ĠD', 'Ġ->', 'ĠA', 'Ġ->', 'ĠE', '.', 'ĠSo', 'ĠE', 'Ġis', 'Ġnow', 'Ġon', 'Ġtop', ',', 'Ġbut', 'Ġoriginally', 'Ġit', 'Ġwas', 'Ġthe', 'Ġbase', '.', 'ĠSo', 'Ġwe', 'Ġneed', 'Ġto', 'Ġmove', 'ĠE', 'Ġfrom', 'Ġthe', 'Ġbottom', 'Ġto', 'Ġthe', 'Ġtop', '.', 'ĠTo', 'Ġdo', 'Ġthat', ',', 'Ġwe', 'Ġhave', 'Ġto', 'Ġdismantle', 'Ġthe', 'Ġexisting']



In [76]:
def section_vector(section: dict, idx: int, layer: int) -> np.ndarray:
    hs = ray.get(hidden_states[idx][layer])
    window_start = section["pos_before"] - 1
    window_end = min(section["pos_after"], section["pos_before"] + 30) 
    # print(section["pos_before"], section["pos_after"], hs.shape, hs[section["pos_before"]:section["pos_after"]].shape)
    return hs[window_start:window_end].mean(axis=0)

In [77]:
section_vectors = {
    cat: []
    for cat in labels_low
}

for idx in trange(1000):
    label = dataset_low[idx]["label"]
    if label is None:
        continue
    sections = parse_sections(dataset_low[idx]["label"], labels_low)
    section_tokens = map_sections_into_tokens(sections, dataset[idx])
    for section in section_tokens:
        if section["pos_after"] > 6000:
            continue
        if section["pos_after"] < section["pos_before"] + 2:
            continue
        vector = section_vector(section, idx, layer)
        section_vectors[section["label"]].append({
            "vector": vector,
            "idx": idx,
            "pos_before": section["pos_before"],
            "pos_after": section["pos_after"],
        })

  0%|          | 0/1000 [00:00<?, ?it/s]

In [78]:
for idx in range(20):
    label = dataset_low[idx]["label"]
    if label is None:
        continue
    sections = parse_sections(dataset_low[idx]["label"], labels_low)
    section_tokens = map_sections_into_tokens(sections, dataset[idx])
    for section in section_tokens:
        if section["pos_after"] > 6000:
            continue
        if section["pos_after"] < section["pos_before"] + 2:
            continue
        if section["label"] == "state-tracking":
            print("*****")
            print(section["content"])
            print()
        

*****
Now, C is on B. Then, we need to stack A on C. But A is on the table. So pick up A and stack on C.

*****
Now, A is on C (which is on B on D). Then, we need to stack E on A. E is on the table (from step 2). So pick up E and stack on A.

*****
Now, E is on A. Then, we need to stack F (which was put down in step 1) on E. So pick up F and stack on E.

*****
After all steps, the stacks are:

- D (table) → B → C → A → E → F.

*****
Starting State:

- Hand: empty

- Blocks on table: B (has F on it), C (has A on it). 

- Stack on C: A → D → E (E is clear)

- Stack on B: F (clear)

*****
New state:

  - Hand holds E.

  - D is on A (which is on C). D is now clear (since E was removed).

  - F is still on B, clear.

  - E is now in hand.

*****
- Hand holds D.

  - A is on C, clear now (since D was removed).

  - F is on B, clear.

  - E is on table.

*****
New state:

  - Hand holds A.

  - C is on the table, clear.

  - F is on B, clear.

  - E and D are on the table.

*****
New state:


In [79]:
{k: len(v) for k,v in section_vectors.items()}

{'initial-state-understanding': 2119,
 'goal-state-understanding': 3375,
 'state-tracking': 2880,
 'action-exploration': 6417}

In [80]:
category_vectors = {}

mean_vectors = {
    cat: np.stack([x["vector"].astype(float) for x in vectors]).mean(axis=0)
    for cat, vectors in section_vectors.items()
}

steering_mean = np.stack([x["vector"].astype(float) for xs in section_vectors.values() for x in xs]).mean(axis=0)

steering_vectors = {
    k: v - steering_mean for k, v in mean_vectors.items()
}


In [89]:
idx = 16

row = dataset[idx]
tokens = tokenize_blocksworld_generation(tokenizer, row)

sections = parse_sections(dataset_low[idx]["label"], labels_low)
section_tokens = map_sections_into_tokens(sections, dataset[idx])
section_tokens[:20]

[{'label': 'initial-state-understanding',
  'pos_before': 906,
  'pos_after': 961,
  'text_pos': 440,
  'content': "- A (table) has F on top, then C on F. So the stack is A-F-C. Since C is clear, there's nothing on top of it.\n- E (table) has D on top. D is clear.\n- B is alone on the table, clear."},
 {'label': 'goal-state-understanding',
  'pos_before': 1056,
  'pos_after': 1216,
  'text_pos': 918,
  'content': "Goal states:\n- Block A is on top of Block D → So A must be on D. But currently, A is on the table. So D must be under A.\n- Block B is on top of Block F → B must be on F. Currently, B is on the table.\n- Block D is on top of Block C → D must be on C. Currently, D is on E, which is on the table. So D needs to be moved to C.\n- Block E is on top of Block B → E must be on B. Currently, E has D on it, so E is under D. So E needs to be moved to B.\n- Block F is on top of Block A → F is already on A, so that's already satisfied. So that part is okay."},
 {'label': 'initial-state-u

In [93]:
# steering_pos = section_tokens[4]["pos_before"] - 1
steering_pos = 2700- 1
steering_vector = steering_vectors["state-tracking"]

In [94]:
print(tokenizer.decode(tokens[0][steering_pos + 1:steering_pos+400]))

 clear.

3. Unstack D from E → hold D.

4. Stack D onto C → D is now on C. Now, D is clear (since nothing is on top of it). Wait, but after stacking, D is on top of C, so C is under D. So D is clear, but C is not.

Now, D is on C. Then, we need to get A on top of D. To do that, A must be moved to D. But A is currently under F (since F was on A, but we unstacked C from F, so F is now on A, but C is gone. Wait, after step 1, when we unstacked C from F, F is still on A, right? Because we only removed C from F. So F is still on A, and F is now clear (since C was on top, but we took it off). So F is clear now.

Wait, let me retrace:

Original stack: A-F-C. After unstacking C from F, the stack becomes A-F (with F now clear, since C was on top and we removed it). Then, we put down C on the table. So now:

- A is on table, with F on it (F is clear).

- C is on table, clear.

- D is on E (still?), no, step 3 was unstack D from E. So after step 3, D is held, and E is now clear (since D was on it

In [101]:
from collections import OrderedDict


def forward_hook(module, input, output):
    """Replace output with the mean representation"""
    output = output[0]

    output[0, -1] += torch.tensor(steering_vector, device=output.device, dtype=torch.float16) * 4

    if output.shape[1] == 1:
        return (output,)
    
    output = output[0]    

    # output[-20:] += torch.tensor(steering_vector, device=output.device, dtype=torch.float16) * 5

    print("asdasd")
    
    print(output.shape, steering_pos)

    return (output.unsqueeze(0),)
    

for m in model.modules():
    m._forward_hooks = OrderedDict()
    
model.model.layers[layer].register_forward_hook(forward_hook)

with torch.no_grad():
    enc = model.generate(tokens[:, :steering_pos-5].to(device), do_sample=False, max_new_tokens=200, temperature=None, top_p=None, top_k=None, use_cache=True).cpu()


asdasd
torch.Size([2694, 5120]) 2699


In [102]:
print(tokenizer.decode(enc[0, steering_pos + 1:steering_pos+200]))

. Now, F is on A, but C is free.

Then, we can proceed to unstack F from A? Wait, F is on A, but it's clear (since nothing is on top of it). Wait, F was on A, but C was previously on F, but now C is on the table. So F is on A, clear.

So:

Step 3: F is on A, clear.

Now, to get F free, we can unstack it from A. But to do that, we need to unstack it, but we can pick it up.

Wait, let's see:

After step 2, F is on A, and C is on the table. So F is clear.

So:

Step 1: unstacked C, put down.

Step , F is on A.

Now, to get F free, we can unstack it from A. But to do that, we can


In [228]:
print(tokenizer.decode(tokens[0][steering_pos - 10 + 1:steering_pos+200]))

 clear since C was removed. Hand holds D.)

4. Put down Block D. (D is on table, clear.)

5. Unstack Block B from on top of Block F. (B is clear now. Hand holds B.)

6. Stack Block B on top of Block E. (E is clear. Hand now empty.)

7. Unstack Block F from on top of Block A. (F is clear now. Hand holds F.)

8. Pick up Block A. (A is on table, clear. Hand holds A.)

9. Stack Block A on top of Block B. (B is clear. Hand empty.)

10. Stack Block F (from hand after step7) on top of Block A. (A is clear now. Hand empty.)

Wait, after step7, F is in hand. After step8, we pick up A, so we have to put down F first? Wait, no. Wait, step7: unstack F from A, so F is in hand. Then step


In [219]:
len(tokenize_blocksworld_generation(tokenizer, row)[0])

6141

In [176]:
len(tokenizer.encode(row["generation"]))

10549

In [177]:
print(row["generation"])

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, let me restate the initial conditions and the goal to make sure I have them right. The initial conditions are:

- Block C craves Block B (so "Object Craves other object" for C and B)
- Harmony exists (Harmony is true)
- Planet Block A, Planet Block B, Planet Block D (so all these blocks are on a planet)
- Province Block A, Province Block C, Province Block D (so these are in a province, but what about Block B? Wait, the initial conditions don't mention province for Block B. Wait, the problem says "province Block A, province Block C and province Block D." So Block B's province status isn't mentioned here. Hmm, maybe it's not in a province? Or maybe it's a typo? Wait, the problem says "province Block A, province Block C and province Block D." So Block B is not in